In [1]:
import requests
import pandas as pd
import uuid

# Basis configuratie gebaseerd op de technische documentatie
BASE_URL = "https://api.ah.nl"
STORE_ID = "1558"  # Dit is het unieke ID voor AH Woenselse Markt Eindhoven

# Verplichte headers om de AH app na te bootsen
headers = {
    "User-Agent": "Appie/9.28 (iPhone17,3; iPhone; CPU OS 26_1 like Mac OS X)",
    "x-application": "AHWEBSHOP",
    "x-clientname": "appie-ios",
    "x-": "9.28",
    "x-fraud-detection-installation-id": str(uuid.uuid4()), # Een unieke ID per sessie
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [2]:
def get_anonymous_token():
    auth_url = f"{BASE_URL}/mobile-auth/v1/auth/token/anonymous"
    payload = {"clientId": "appie-ios"}
    
    response = requests.post(auth_url, json=payload, headers=headers)
    response.raise_for_status() # Geeft een foutmelding als het misgaat
    
    token_data = response.json()
    return token_data['access_token']

# Activeer de sleutel voor alle volgende verzoeken
access_token = get_anonymous_token()
headers["Authorization"] = f"Bearer {access_token}"
print("Handshake succesvol: Token opgehaald.")

Handshake succesvol: Token opgehaald.


In [3]:
bargain_query = """
query GetBargains($storeId: String!) {
  bargainItems(storeId: $storeId) {
    categoryTitle  # <--- Deze voegt de categorie (zoals Vlees) toe
    product {
      title
      brand
      salesUnitSize
    }
    bargainPrice {
      priceWas
      priceNow
    }
    markdown {
      markdownPercentage
      markdownExpirationDate
    }
    stock
  }
}
"""

def fetch_laatste_kans(store_id):
    url = f"{BASE_URL}/graphql"
    
    graphql_headers = headers.copy()
    graphql_headers.update({
        "x-apollo-operation-name": "GetBargains",
        "x-apollo-operation-type": "query",
        "apollographql-client-name": "nl.ah.Appie-apollo-ios",
        "apollographql-client-version": "9.28-260102201630"
    })
    
    payload = {
        'query': bargain_query, 
        'variables': {'storeId': store_id},
        'operationName': 'GetBargains'
    }
    
    response = requests.post(url, json=payload, headers=graphql_headers)
    data = response.json()
    
    if 'errors' in data:
        print("❌ GraphQL Foutmelding gevonden:")
        for error in data['errors']:
            print(f" - {error.get('message')}")
        return None
        
    return data.get('data', {}).get('bargainItems')

# Haal de ruwe data opnieuw op
raw_items = fetch_laatste_kans(STORE_ID)

if raw_items:
    print(f"✅ Succes! {len(raw_items)} producten gevonden op de Woenselse Markt.")
else:
    print("❌ Geen data ontvangen. Controleer de output hierboven.")

✅ Succes! 174 producten gevonden op de Woenselse Markt.


In [5]:
# Gebruik json_normalize om geneste velden (zoals product.title) plat te slaan
df = pd.json_normalize(raw_items)

# Optioneel: Kolomnamen opschonen voor gemak
df.columns = [c.replace('product.', '').replace('bargainPrice.', '').replace('markdown.', '') for c in df.columns]

# Sorteer op de hoogste korting
# df = df.sort_values(by='markdownPercentage', ascending=False)

# Toon de live status
display(df.head(10))

,categoryTitle,stock,priceWas,priceNow,markdownPercentage,markdownExpirationDate,title,brand,salesUnitSize
0,"Groente, aardappelen",2,1.99,1.19,40,2026-01-26,Bieze Rauwkost komkommer,Bieze,250 g
1,"Groente, aardappelen",5,1.39,1.04,25,2026-01-26,AH Rucola slamelange,AH,75 g
2,"Groente, aardappelen",5,1.69,1.27,25,2026-01-26,AH Aardappelen voor de stamppot,AH,1 kg
3,"Groente, aardappelen",4,1.69,1.27,25,2026-01-26,AH Hutspot,AH,500 g
4,"Groente, aardappelen",4,2.99,2.24,25,2026-01-26,AH Roerbakgroente Italiaans fijngesneden,AH,400 g
5,"Groente, aardappelen",3,1.99,1.49,25,2026-01-26,AH Taugé grootverpakking,AH,250 g
6,"Groente, aardappelen",3,2.39,1.79,25,2026-01-26,AH Biologisch Vastkokende aardappelen,AH Biologisch,1 kg
7,"Groente, aardappelen",2,3.99,1.79,25,2026-01-26,AH Stoomgroente bloemkool marscarponesaus,AH,450 g
8,"Groente, aardappelen",2,1.59,1.19,25,2026-01-26,AH Eikenblad slamelange,AH,100 g
9,"Groente, aardappelen",2,4.49,3.37,25,2026-01-26,AH Pompoensoep verspakket,AH,6 pers | 35 min
